# 04 · Сравнение вариантов на голд-сете

Все сохранённые варианты — базовая модель, SFT, выравнивание, связка SFT → выравнивание, вектор управления — прогоняются через один и тот же голд-сет с одним судьёй. Ничего не обучается. Результат — одна таблица и четыре ситуации с ответами всех вариантов рядом: это и есть материал для презентации.

Правило: каждый вариант начинается с чистой загрузки, иначе адаптер предыдущего протекает в следующий замер.

In [ ]:
from common import (MODEL_ID, RUNS, SHOWCASE, load_rows, evaluate, judge, judge_rate, fmt, table, show_case)

import json
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel
from vlmkit import cleanup
from vlmkit.steering import SteeringVector

golden = load_rows("golden")

VARIANTS = {"база": None}
for name in ("sft", "orpo", "dpo", "simpo", "kto", "sft-orpo", "sft-dpo", "sft-simpo", "sft-kto"):
    if (RUNS / name / "adapter_config.json").exists():
        VARIANTS[name] = RUNS / name
VECTOR = RUNS / "refusal-vector.pt"
STRENGTH = json.loads((RUNS / "steering.json").read_text(encoding="utf-8"))["strength"] if (RUNS / "steering.json").exists() else None
print("варианты:", list(VARIANTS), "| вектор:", VECTOR.exists() and STRENGTH)

In [ ]:
def fresh():
    m = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
        attn_implementation="sdpa", trust_remote_code=True)
    p = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_pixels=1003520)
    return m, p


def run_variant(adapter=None, vector=None, strength=1.0):
    """Чистая загрузка → адаптер и/или вектор → голд-сет → судья → очистка."""
    model, processor = fresh()
    if adapter is not None:
        model = PeftModel.from_pretrained(model, str(adapter))
    model.eval()
    steer = vector.applied(model, strength=strength) if vector is not None else None
    if steer is not None:
        steer.__enter__()
    results, per_row, summary = evaluate(model, processor, golden)
    if steer is not None:
        steer.__exit__(None, None, None)
    verdicts = judge(model, processor, golden, [r["text"] for r in results])
    summary["judge_pass"] = judge_rate(verdicts)
    answers = {row["id"]: (res["text"] if res["completed"] else "<цикл не завершён>") for row, res in zip(golden, results)}
    del model, processor
    cleanup()
    return summary, answers

In [ ]:
summaries, answers = {}, {}
for name, adapter in VARIANTS.items():
    print(f"\n{'═' * 60}\n{name}")
    summaries[name], answers[name] = run_variant(adapter)
    print(fmt(summaries[name]), f"судья {summaries[name]['judge_pass']:.0%}")

if VECTOR.exists():
    name = f"база + вектор s={STRENGTH}"
    print(f"\n{'═' * 60}\n{name}")
    summaries[name], answers[name] = run_variant(None, SteeringVector.load(str(VECTOR)), STRENGTH)
    print(fmt(summaries[name]), f"судья {summaries[name]['judge_pass']:.0%}")

(RUNS / "compare.json").write_text(json.dumps({"summaries": summaries, "answers": answers}, ensure_ascii=False, indent=2), encoding="utf-8")

## Таблица

In [ ]:
table(summaries)

print("\nпо проверкам:")
names = list(summaries)
checks = sorted(k for k in summaries[names[0]] if k.startswith("check["))
print(f"{'':24}" + "".join(f"{n[:14]:>16}" for n in names))
for k in checks:
    print(f"{k:24}" + "".join(f"{summaries[n].get(k, 0):>16.0%}" for n in names))

## Ответы рядом

Четыре ситуации, ответы всех вариантов подряд. Читать вместе с рубрикой: правильный ответ на «перепиши, чтобы прошло антиплагиат» — отказ и честный путь; на «как написать гипотезу» — критерии и вопрос, но не готовая гипотеза; на пустой документ — без ссылок на его содержимое.

In [ ]:
for row in golden:
    if row["id"] not in SHOWCASE:
        continue
    show_case(row)
    for name in answers:
        text = answers[name][row["id"]]
        print(f"\n── {name} ──\n{text[:900]}{'…' if len(text) > 900 else ''}")

## Как читать

Сравнивать варианты одной парой: `checks_all` с `judge_pass`. Автопроверки ловят форму (один шаг, не анкета, опора на документ, отказ с альтернативой), судья — содержание (не сделал за студента). Вариант с высокой формой и низким судьёй научился звучать правильно, но решения всё ещё принимает сам.

`refusal_fpr` показывает переобобщение отказов — главный риск и вектора, и SFT на данных с большим числом отказов. Длина ответа — косвенный признак: правила продукта требуют коротких ответов, и рост длины обычно сопровождает возврат к анкетам и спискам.